# Lecture 4 — Judge probability forecasts by more than accuracy

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opherdonchin/AdvancedStatisticsCourse_36725561/blob/main/notebooks/lecture4_iris_forecasts.ipynb)


In the lecture, a classifier fitted to the Iris flowers' sepal measurements gave each flower a forecast $q(Y\mid x)$: a probability for each of the three species. On simulated flowers we knew the true distribution $p^*$ and could compute expected log scores exactly. Real flowers give us only one label each. In this notebook we score real forecasts on 75 flowers the fit never saw.

**Is the classifier's forecast better than a forecast that ignores the sepals, by which measure, and is it too sure or not sure enough?**

| Section | What we do | Contribution to the question |
|---|---|---|
| 1. Accuracy | Predicted species and accuracy for three forecasts | What accuracy can see |
| 2. Log score | The mean log score of each forecast on the held-out flowers | What accuracy cannot see |
| 3. Is the difference real? | Per-flower differences, their standard error, and four individual flowers | Whether a difference in mean log score is more than noise, and where it comes from |
| 4. Calibration | Confidence against accuracy, in bins | Whether the forecast is too sure or not sure enough |

### *Explanation to read*

#### The classifier: multinomial logistic regression

The forecast `q` comes from *multinomial logistic regression*, the softmax classifier of the lecture. For an input $x$ (here the two sepal measurements) it computes one logit per species, a weighted sum of the measurements plus an offset, $z_y(x)=w_y^\top x+b_y$, and forecasts $q(y\mid x)=\mathrm{sm}(z(x))_y$. *Fitting* chooses the weights $w_y$ and offsets $b_y$ that make the training flowers' actual species probable: it maximizes the likelihood of the training labels, less a penalty on large weights that scikit-learn applies by default. How and why this works is the subject of the next lecture (L05); here we only use the fitted forecasts.

In scikit-learn the model is used in three steps:

- `LogisticRegression()` creates an unfitted model with the default settings.
- `classifier.fit(X, labels)` fits it. `X` has one row per training flower and one column per measurement (here shape `(75, 2)`), and `labels` holds their species codes 0, 1, 2 (shape `(75,)`). It returns the fitted model.
- The fitted model forecasts new inputs `X_new` of shape `(n, 2)`. `classifier.decision_function(X_new)` returns the logits $z(x)$, shape `(n, 3)`; `classifier.predict_proba(X_new)` returns their softmax $q(y\mid x)$, each row summing to 1; `classifier.predict(X_new)` returns the most probable species. The columns follow `classifier.classes_`, here `[0, 1, 2]`.

The set-up below uses `decision_function`, so that it can also divide the logits by a temperature.

### *Code to run*

#### Set up the flowers and the three forecasts

This cell loads the 150 Iris flowers and keeps sepal length and width, the lecture's input $x$. It splits the flowers into 75 training and 75 held-out flowers, 25 of each species in each half (seed 2027, fixed before anything was scored); an even split leaves many held-out flowers to score each forecast on. The classifier is fitted to the training flowers only.

It produces three forecast tables for the held-out flowers. Each is an array of shape `(75, 3)`: row $m$ is the forecast for held-out flower $m$, and the columns are setosa, versicolor, virginica, in that order.

- `q`: the fitted classifier's forecast, $q(y\mid x_m)=\mathrm{sm}(z(x_m))_y$, the softmax of its logits $z(x_m)$, as in the lecture.
- `q2`: the same classifier at temperature $T=2$, $q_2(y\mid x_m)=\mathrm{sm}(z(x_m)/2)_y$.
- `q0`: the baseline $q_0(y)$, the species frequencies among the training flowers. It is the same row for every flower: it ignores the sepals.

`y` holds the held-out flowers' species (0 setosa, 1 versicolor, 2 virginica), `M` their number, and `flower` their flower numbers (rows of the 150-flower table counted from 1, as in the lecture).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import softmax
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

iris = load_iris()
sepals, labels = iris.data[:, :2], iris.target
species = list(iris.target_names)
train, test = train_test_split(np.arange(150), test_size=75, random_state=2027, stratify=labels)

# Preparation: the classifier is fitted to the training flowers only (fitting is L05's subject).
classifier = LogisticRegression().fit(sepals[train], labels[train])
logits = classifier.decision_function(sepals[test])        # z(x_m), shape (75, 3)

q = softmax(logits, axis=1)                                # fitted forecast
q2 = softmax(logits / 2, axis=1)                           # the same classifier at T = 2
q0 = np.tile(np.bincount(labels[train]) / len(train), (75, 1))   # training species frequencies

y = labels[test]
M = len(y)
flower = test + 1
print("forecast q for held-out flower", flower[0], ":", np.round(q[0], 3), "  its species:", species[y[0]])

### *Code to run*

#### Check that every row is a probability distribution

Before scoring, we check each table: 75 rows, no negative entry, and every row summing to 1 (`np.allclose` allows for rounding).

In [ ]:
for name, forecast in [("q", q), ("q2", q2), ("q0", q0)]:
    assert forecast.shape == (M, 3)
    assert np.all(forecast >= 0) and np.allclose(forecast.sum(axis=1), 1)
print("q, q2 and q0: 75 rows each, every row a probability distribution over the three species")

## 1. Accuracy

### *Explanation to read*

#### Predicted species and accuracy

A forecast's predicted species for flower $m$ is its most probable species, $\hat y_m=\arg\max_y q(y\mid x_m)$, the lecture's rule for the classification task. `np.argmax(q, axis=1)` returns, for each row of `q`, the column of its largest entry. When several entries tie it returns the first, so `q0`, whose three entries are equal, predicts setosa (column 0) for every flower; that is our tie rule. Accuracy is the fraction of flowers whose predicted species is their actual species $y_m$.

### *Questions to answer*

#### 1. Predicted species and accuracy — code

Compute the predicted species of `q`, `q2` and `q0` and print the accuracy of each. Then check whether `q` and `q2` predict the same species for every flower. (Check: `q0` predicts setosa for all 75 flowers, and 25 of them are setosa, so its accuracy must be 1/3.)

In [ ]:
# answer here

### *Explanation to read*

#### Section summary

The classifier `q` predicts 62 of the 75 held-out flowers correctly (accuracy 0.827); the baseline `q0` gets the 25 setosa right (1/3). `q` and `q2` predict exactly the same species for every flower, so they have the same accuracy: dividing the logits by $T$ never changes which species is most probable (lecture, Section 1). Accuracy cannot tell `q` from `q2`, although their probabilities differ.

## 2. Log score

### *Explanation to read*

#### The log score on held-out flowers

The lecture scored a forecast by the probability it gave to what happened. For held-out flower $m$, whose species is $y_m$, the log score is

$$
\ell_m=-\ln q(y_m\mid x_m), \tag{1}
$$

in nats. Lower is better: a forecast that gave the actual species probability 0.9 pays 0.105, one that gave it 0.1 pays 2.303. The mean over the $M$ held-out flowers,

$$
\hat S=\frac{1}{M}\sum_{m=1}^{M}-\ln q(y_m\mid x_m), \tag{2}
$$

estimates the forecast's expected log score on new flowers. It is right on average because the held-out flowers were not used by the fit (lecture, Section 2: a mean log score on held-out cases estimates the expected log score). Its error is

$$
\sigma/\sqrt{M}, \tag{3}
$$

where $\sigma$ is the standard deviation of one flower's score. We estimate $\sigma$ by the sample standard deviation of the 75 scores.

### *Explanation to read*

#### Picking each flower's probability for its own species

Equation (1) needs one entry from each row of `q`: in row $m$, the column $y_m$. NumPy picks one entry per row in a single step when we index with two integer arrays of the same length: `q[rows, cols]` returns `q[rows[0], cols[0]]`, `q[rows[1], cols[1]]`, and so on. This is *integer array indexing* ([NumPy documentation](https://numpy.org/doc/2.0/user/basics.indexing.html#integer-array-indexing)). For example, `q[[4, 0, 9], [2, 0, 1]]` returns the three numbers `q[4, 2]`, `q[0, 0]` and `q[9, 1]`. It is different from `q[0:3, [2, 0, 1]]`, which mixes a slice with an array and returns a $3\times3$ block.

### *Code to run*

#### Integer array indexing on three flowers

`sample` holds the positions of the first held-out setosa, versicolor and virginica. The cell prints their rows of `q` and their species, and applies integer array indexing to those three rows.

In [ ]:
sample = np.array([np.flatnonzero(y == k)[0] for k in range(3)])
print("q[sample], one row per flower:")
print(np.round(q[sample], 3))
print("species y[sample]:", y[sample])
print("q[sample, [2, 0, 1]]:", np.round(q[sample, [2, 0, 1]], 3))

### *Questions to answer*

#### 2. Each flower's probability for its own species — code

Use integer array indexing to make `p_obs`, an array of length 75 holding $q(y_m\mid x_m)$ for every held-out flower: the probability `q` gave each flower's actual species. Print `p_obs[sample]` and check its three entries against the three rows and species printed above.

In [ ]:
# answer here

#### 3. Mean log score and its standard error — code

Write `log_scores(forecast)`, returning the 75 log scores (1) of a forecast table. Use it to print the mean log score $\hat S$ (2) and its standard error (3) for `q`, `q2` and `q0`. Use `np.std(..., ddof=1)` for the sample standard deviation. (Check: `q0` gives every flower's species probability 1/3, so each of its scores is $\ln 3=1.099$ and its standard error is 0.)

In [ ]:
# answer here

#### 4. The gain over the baseline — written answer

The fitted forecast `q` improves on the baseline `q0` by about 0.62 nats of held-out mean log score. The lecture showed that the gain of a fitted forecast $q(Y\mid X)$ over a fitted baseline $q_0(Y)$ is

$$
I(X;Y)+\mathrm{KL}(p^*_Y\,\Vert\,q_0)-E_X\,\mathrm{KL}\big(p^*(Y\mid X)\,\Vert\,q(Y\mid X)\big),
$$

where $p^*_Y$ is the true species distribution and $p^*(Y\mid X)$ the true distribution of species given the sepals. Here $q_0=(1/3,1/3,1/3)$, the species proportions of this collection; take new flowers to come in the same proportions, so that $q_0=p^*_Y$. Is 0.62 nats an estimate of the mutual information $I(X;Y)$ between sepals and species? Say in words what it estimates.

- answer here

### *Explanation to read*

#### Section summary

The mean log score sees what accuracy did not: `q` scores 0.482 nats and `q2` 0.630, although they predict the same species, and both are far below the baseline's $\ln 3=1.099$. Each mean is an estimate with a standard error (3), and the gain over the baseline estimates the gain of these fitted forecasts, not the mutual information.

## 3. Is the difference real?

### *Explanation to read*

#### Comparing two forecasts on the same flowers

Question 3 gave one held-out mean log score (2) for each forecast. Write them $\hat S(q)=0.482$ and $\hat S(q_2)=0.630$ nats. Lower is better, so `q2` looks worse by $0.630-0.482=0.148$ nats. Is that difference real, or could it come from which 75 flowers happened to be held out?

Each $\hat S$ is an estimate. A different set of 75 held-out flowers would give somewhat different values, and the standard error (3) says by how much: about 0.055 for `q` and 0.034 for `q2`. If the two estimates were unrelated, the error of their difference would combine both, $\sqrt{0.055^2+0.034^2}\approx0.065$, and 0.148 would be only a little over two such errors.

But the two estimates are not unrelated: both forecasts were scored on the same 75 flowers. A flower that is hard to forecast, such as a versicolor whose sepals look like a virginica's, gets a high score from both `q` and `q2`. With a harder set of held-out flowers both means would rise together, and their difference would change much less than either mean does.

So we compare the forecasts flower by flower. For each held-out flower $m$, the difference of its two log scores (1) is

$$
d_m=-\ln q_2(y_m\mid x_m)-\big(-\ln q(y_m\mid x_m)\big),
$$

which is positive when `q2` did worse on flower $m$. The mean of the 75 differences, $\bar d$, equals the difference of the two means, $\hat S(q_2)-\hat S(q)$. The $d_m$ are one number per held-out flower, just like the scores, so the lecture's reasoning for (3) applies to them: $\bar d$ estimates the expected difference on new flowers, with error

$$
\mathrm{sd}(d)/\sqrt{M}, \tag{4}
$$

where $\mathrm{sd}(d)$ is the sample standard deviation of the 75 differences. A hard flower raises both of its scores, and much of that rise cancels in $d_m$ (here a flower's two scores have correlation 0.97). The differences therefore vary less than the scores, and (4) can be much smaller than 0.065. This is a *paired* comparison. As a rough guide, a $\bar d$ more than about two standard errors from 0 is unlikely to come only from which flowers happened to be held out.

### *Questions to answer*

#### 5. The paired difference — code

Compute the differences $d_m$ of `q2`'s scores minus `q`'s, using your `log_scores`. Print their mean, its standard error (4), and on how many flowers each forecast has the lower score. In a comment, say whether the difference (`q2` scores worse) is larger than its noise.

In [ ]:
# answer here

### *Code to run*

#### Four held-out flowers under both forecasts

Flowers 107 and 51 are mistakes of both forecasts; flowers 71 and 23 are predicted correctly. For each, the table shows its species, the predicted species, both forecasts (setosa, versicolor, virginica) and their log scores (1).

In [ ]:
print(f"{'flower':>6}  {'species':<11} {'predicted':<11} {'q':<21} {'-ln q':>6}   {'q2':<21} {'-ln q2':>6}")
for number in [107, 51, 71, 23]:
    m = np.flatnonzero(flower == number)[0]
    print(f"{number:>6}  {species[y[m]]:<11} {species[np.argmax(q[m])]:<11} {str(np.round(q[m], 3)):<21} "
          f"{-np.log(q[m, y[m]]):>6.3f}   {str(np.round(q2[m], 3)):<21} {-np.log(q2[m, y[m]]):>6.3f}")

### *Questions to answer*

#### 6. Where the difference comes from — written answer

On which of the four flowers does `q2` score better than `q`, and on which worse? Explain why from what temperature $T=2$ does to a forecast, and why `q2`'s mean log score ends higher (worse) although it scores better on some flowers.

- answer here

### *Explanation to read*

#### Section summary

Scored on the same flowers, `q2` is worse than `q` by 0.148 nats, about six standard errors (4): a real difference, not noise. Flattening helps on a few flowers, mistakes such as 107 and 51, and hurts on the many correct ones. The log score showed that `q` is better than `q2`; it did not show whether `q` itself is too sure or not sure enough.

## 4. Calibration

### *Explanation to read*

#### Confidence against accuracy

Sections 2 and 3 showed that `q` is better than `q2`, but not whether `q` itself is too sure or not sure enough. That is a question about *calibration*. Murphy (Book 2, §14.2.2) calls a forecast calibrated when its probabilities match how often things happen: "if a classifier predicts $p(y = c\mid x) = 0.9$, then we expect this to be the true label about 90% of the time."

**What we check.** We use the simplest version: the probability a forecast gives to its own predicted species $\hat y_m$. For flower $m$ this is the *top-label confidence*

$$
\hat p_m=\max_y q(y\mid x_m). \tag{5}
$$

If `q` is calibrated, its predicted species should be right about as often as $\hat p_m$ says.

**Why bins.** One flower cannot test this: its prediction is either right or wrong, and a single outcome says little about a probability of, say, 0.7. So we pool flowers with similar confidence. We split the confidence scale into $B=5$ bins of width 0.2: bin $b$ holds the flowers whose $\hat p_m$ lies in $\big((b-1)/5,\ b/5\big]$. (With three species $\hat p_m\ge1/3$, so the first bin is always empty.) For the flowers in bin $B_b$ we compute two averages (Book 2, Eqs. 14.17–14.18), the fraction predicted correctly and their mean confidence:

$$
\mathrm{acc}(B_b)=\frac{1}{|B_b|}\sum_{m\in B_b}\mathbb{1}(\hat y_m=y_m),\qquad
\mathrm{conf}(B_b)=\frac{1}{|B_b|}\sum_{m\in B_b}\hat p_m . \tag{6}
$$

**How to read them.** For a calibrated forecast the two are about equal in every bin. Accuracy above confidence means the forecast was right more often than it claimed: it is not sure enough (*under-confident*). Accuracy below confidence means it is too sure (*over-confident*). A plot of the two, bin by bin, is a *reliability diagram*.

**How big a gap matters.** Even a calibrated forecast would not match exactly, because each bin holds only about 25 flowers. Take a bin with $n$ flowers and mean confidence $c$. If the forecast is calibrated, each of these flowers is predicted correctly with probability about $c$, so the number predicted correctly is roughly binomial, and the bin's accuracy varies around $c$ with standard error

$$
\sqrt{c(1-c)/n}, \tag{7}
$$

about 0.09 for $c=0.75$ and $n=25$. As in Section 3, a gap of more than about two standard errors is unlikely to be chance alone.

**What the check does not look at.** It uses only the probability given to the predicted species, not the probabilities given to the other two.

### *Questions to answer*

#### 7. The top bin by hand — code

For `q`, compute the array `confidence` of $\hat p_m$ (5) and the boolean array `correct`, true where $\hat y_m=y_m$. Then, for the top bin, $\hat p_m\in(0.8,1]$, print the number of flowers, their accuracy and their mean confidence (6). (A boolean array selects entries: `correct[confidence > 0.8]` keeps the entries of `correct` where the confidence is above 0.8.)

In [ ]:
# answer here

### *Code to run*

#### Reliability diagrams for `q` and `q2`

`reliability(forecast)` computes (6) for every bin of a forecast table that holds at least one flower. The cell prints each bin with its binomial standard error (7), and draws both diagrams on the same axes: bars are the accuracy, dots the mean confidence, and the dashed diagonal is where they would be equal. Its top bin for `q` should match your answer to Question 7.

In [ ]:
def reliability(forecast, B=5):
    confidence = forecast.max(axis=1)
    correct = forecast.argmax(axis=1) == y
    bin_index = np.ceil(confidence * B).astype(int) - 1     # index b - 1 for the bin ((b - 1)/B, b/B]
    table = []
    for b in range(B):
        members = bin_index == b
        if members.any():
            table.append((b / B, (b + 1) / B, members.sum(), confidence[members].mean(), correct[members].mean()))
    return table


fig, axes = plt.subplots(1, 2, figsize=(9, 3.8), sharey=True)
for ax, name, forecast in [(axes[0], "q (fitted)", q), (axes[1], "q2 (T = 2)", q2)]:
    print(name)
    for low, high, n, conf, acc in reliability(forecast):
        error = np.sqrt(conf * (1 - conf) / n)
        print(f"  ({low:.1f}, {high:.1f}]: n = {n:2d}, mean confidence {conf:.3f}, accuracy {acc:.3f}, standard error {error:.3f}")
        ax.bar((low + high) / 2, acc, width=0.18, color="C0", alpha=0.75)
        ax.plot((low + high) / 2, conf, "o", color="C3")
        ax.text((low + high) / 2, 0.03, f"n={n}", ha="center", fontsize=8, color="white")
    ax.plot([0, 1], [0, 1], "--", color="0.6")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.05)
    ax.set_title(name)
    ax.set_xlabel("top-label confidence")
axes[0].set_ylabel("accuracy (bars), mean confidence (dots)")
plt.show()

### *Questions to answer*

#### 8. Too sure or not sure enough? — written answer

(a) From the printed bins and the diagrams, is `q` too sure or not sure enough? Is `q2`?
(b) For `q`, compare each bin's gap between accuracy and confidence with its printed standard error. How strong is the evidence, and what does this check not look at?

- answer here

### *Explanation to read*

#### Section summary

Grouping the held-out flowers by top-label confidence showed what the mean log score could not: the direction. The fitted forecast `q` is somewhat under-confident, with accuracy at or above confidence in every bin, and `q2` is more so. With about 25 flowers per bin, the largest gaps are nearly two standard errors, so the evidence is moderate, and the check looks only at the predicted species.

### *Explanation to read*

#### Answer to the opening question

**Better than the baseline, by both measures.** On 75 held-out flowers the classifier's forecast `q` predicts 82.7% of species correctly against the baseline's 33.3%, and its mean log score is 0.482 nats against $\ln 3=1.099$. The gain of 0.62 nats estimates the gain of these fitted forecasts, not the mutual information between sepals and species.

**Which measure.** Accuracy sees only the predicted species: `q` and its flattened version `q2` tie at 0.827. The log score separates them by 0.148 nats, about six standard errors when the two are compared on the same flowers.

**Too sure or not sure enough.** The reliability diagram says `q` is somewhat under-confident: its accuracy is at or above its confidence in every bin. The evidence is moderate (75 flowers, top label only). The fitting procedure we used gives under-confident forecasts on these flowers; L05 develops that procedure and its logic next week.

**Sources.** Murphy, *Probabilistic Machine Learning: Advanced Topics*, §14.2.2 and §14.2.2.1, pp. 578–579 (calibration; Eqs. 14.17–14.18). Iris data: Fisher (1936), via scikit-learn's `load_iris`. The split, the classifier's settings, the temperature variant and the bins are course choices.